# PhishARG — Reentrenamiento con Corpus Curado v3 (Colab A100 GPU)

**Proyecto de Tesis:** PhishARG — Detección de Phishing con IA Explicable

**Autores:** Gabriel Adia & Tomás Basualdo | **Universidad:** UADE

Este notebook implementa las 4 mejoras Data-Centric identificadas tras auditar el modelo `multilingual-candidate-a100`:
1. **Exclusión de ruido y etiquetas inconsistentes** (5 casos identificados en auditoría).
2. **Inyección de pares contrastivos semánticos bilingües** (diferenciación consejo defensivo vs ataque activo).
3. **Amenazas modernas 2026** (Device Code, MFA push fatigue, redirección de pagos BEC, Quishing).
4. **Preservación estricta de partición agrupada** (0 data leakage garantizado con splits v3).

> **NOTA IMPORTANTE:** Antes de comenzar, asegúrate de activar la GPU A100 en `Entorno de ejecución` > `Cambiar tipo de entorno de ejecución` > **GPU A100** > Guardar.


## Paso 1: Clonar el repositorio y cambiar a la rama de trabajo

In [ ]:
# Clonar repositorio y activar rama codex/hybrid-nlp
!git clone https://github.com/gabrieladia1979/flujo_v2.git
%cd flujo_v2
!git checkout codex/hybrid-nlp
!git pull origin codex/hybrid-nlp
print("Repositorio actualizado en rama codex/hybrid-nlp.")

## Paso 2: Instalar dependencias y verificar acelerador GPU

In [ ]:
!pip install -q sentence-transformers==5.7.0 xgboost scikit-learn

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"Memoria VRAM: {mem:.1f} GB")
else:
    print("ADVERTENCIA: No hay GPU activa. Se recomienda A100 para máxima velocidad.")

## Paso 3: Cargar el corpus curado v3

Tienes dos opciones:
- **Opción recomendada (Rápida - 7.5 MB):** Subir `artifacts/hybrid/colab-curated-v3.zip` generado en tu PC.
- **Opción alternativa:** Si subes `multilingual-v1.csv` a `artifacts/hybrid/corpus/`, el script lo curará automáticamente.

In [ ]:
import json
import zipfile
from pathlib import Path
from google.colab import files

corpus_path = Path("artifacts/hybrid/corpus-v3/training.csv")

if not corpus_path.exists():
    print("Sube el archivo 'colab-curated-v3.zip' (7.5 MB) desde tu PC (flujo_v2/artifacts/hybrid/):")
    uploaded = files.upload()
    for fn in uploaded.keys():
        if fn.endswith('.zip'):
            with zipfile.ZipFile(fn, 'r') as zip_ref:
                zip_ref.extractall('.')
            print(f"Extracción completada de {fn}.")

manifest_path = Path("artifacts/hybrid/corpus-v3/manifest.json")
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    print("\nManifest del corpus curado v3 verificado:")
    for k, v in manifest.items():
        print(f"  {k}: {v}")
else:
    print("Generando corpus v3 localmente en el entorno...")
    !python scripts/build_curated_corpus_v3.py

## Paso 4: Entrenamiento con A100 (5 Épocas, 384 Tokens, 4 Capas)

Entrenamos el modelo con la configuración de máxima potencia validada en los experimentos de ablación.

In [ ]:
!python scripts/train_hybrid.py \
  --dataset artifacts/hybrid/corpus-v3/training.csv \
  --previous-splits artifacts/hybrid/corpus-v3/splits.json \
  --phishing-label 1 \
  --output artifacts/hybrid/multilingual-candidate-v3-curated \
  --report reports/hybrid_multilingual_v3_curated \
  --epochs 5 \
  --batch-size 64 \
  --max-tokens 384 \
  --trainable-layers 4 \
  --device cuda

## Paso 5: Evaluación exacta sobre holdout y diagnóstico contrastivo

In [ ]:
!python scripts/compare_holdout_exact.py \
  --model artifacts/hybrid/multilingual-candidate-v3-curated \
  --splits artifacts/hybrid/corpus-v3/splits.json \
  --dataset artifacts/hybrid/corpus-v3/training.csv \
  --contrastive data/hybrid_contrastive_curated_v3.jsonl \
  --output reports/hybrid_v3_curated_exact_eval

## Paso 6: Ver métricas y comparación

In [ ]:
import json
from pathlib import Path

rep_file = Path("reports/hybrid_v3_curated_exact_eval.json")
if rep_file.exists():
    data = json.loads(rep_file.read_text())
    print(json.dumps(data, indent=2))
else:
    print("Evaluación finalizada.")

## Paso 7: Empaquetar y descargar el modelo entrenado

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("multilingual-candidate-v3-curated", "zip", "artifacts/hybrid/multilingual-candidate-v3-curated")
shutil.make_archive("reports_v3_curated", "zip", "reports")

print("Descargando artefactos...")
files.download("multilingual-candidate-v3-curated.zip")
files.download("reports_v3_curated.zip")
print("Descarga iniciada. Para instalar en Windows:\n")
print("1. Descomprimir en artifacts/hybrid/multilingual-candidate-v3-curated/")
print("2. Ejecutar: .venv-hybrid/Scripts/python.exe scripts/compare_holdout_exact.py --model artifacts/hybrid/multilingual-candidate-v3-curated ...")